# 05 確率予測 — 「何kWhか」ではなく「どのくらいの範囲に入るか」

> 🔒 すべて**合成データ**です。⏱ 所要時間: 説明 15分 ＋ 実行 数秒（比較表は事前に `src/run_phase5.py`（約10分）を実行した結果のCSVを読みます）

## ゴール
- 点の予測（1つの値）ではなく、**予測区間**（例: 80%の確率で この範囲）を出す方法を知る。
- **pinball loss**（分位点損失）で、ニューラルネットや勾配ブースティングに分位点を学習させる。
- 区間が**正直か**（80%区間に、実測が本当に80%入るか）を**カバー率**で確かめ、外れていたら**校正**（CQR）する。
- 不確かさが、**リード**（先の日ほど）と**天気**（曇りで）でどう変わるかを見る。

## なぜ区間か
発電量の予測は、「明日は10 kWh」より「**8〜12 kWhに入る（80%）**」のほうが、**蓄電池の運用や電力の調達**の判断に役立ちます。不確かさの大きさ自体が情報です。

## キーワード
- **分位点**: 例えば0.9分位点は「実測の90%がこれ以下」となる値。10%点〜90%点で**80%区間**。
- **pinball loss**: 分位点 τ の予測 q、実測 y に対し `max(τ(y−q), (τ−1)(y−q))`。τ=0.9 なら「実測が上回る」ことを重く罰するので、予測は上位10%の位置に寄る。
- **カバー率**: 実測が区間に入る割合。80%区間なら **0.8に近いほど正直**。足りなければ、区間が狭すぎる（自信過剰）。
- **区間の幅**: 狭いほど情報があるが、**カバー率が足りないなら意味がない**（先にカバー率）。
- **校正（CQR）**: 校正用データで、区間の幅を足し引きして、カバー率を合わせる。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]
pd.set_option("display.width", 220)
import run_phase5
from tsf_synth import make_series, split_series
from tsf_models import mlp_fit_predict

# 1つの世界で、MLPの分位点予測を1年分描く
sp = split_series(make_series(1_000_001)); tr, va = sp["train"], sp["val"]
Q = mlp_fit_predict(tr, va, 1, groups=("base",), hidden=(32,), n_seeds=3, quantiles=run_phase5.QS)
d = va.iloc[:120]
fig, ax = plt.subplots(figsize=(13, 4))
ax.fill_between(d["date"], Q[:120, 0], Q[:120, 8], alpha=0.25, label="80%区間（10%点〜90%点）")
ax.fill_between(d["date"], Q[:120, 2], Q[:120, 6], alpha=0.35, label="40%区間（30%点〜70%点）")
ax.plot(d["date"], Q[:120, 4], lw=1, label="中央値（50%点）")
ax.plot(d["date"], d["gen_kwh"], "k.", ms=4, label="実測")
ax.set_ylabel("発電量 kWh/日"); ax.legend(ncol=4, fontsize=8); ax.grid(alpha=0.3)
inside = ((va["gen_kwh"] >= Q[:, 0]) & (va["gen_kwh"] <= Q[:, 8])).mean()
ax.set_title(f"MLP分位点予測（L=1・検証の最初の120日）。検証2年全体で、80%区間に入った割合 = {inside:.3f}")
plt.tight_layout(); plt.show()

**出力の見方**: 帯が予測区間、黒点が実測です。**実測の大半が80%区間（濃い外側の帯）に入り**、内側の40%区間には約4割が入る、というのが「正直な」区間です。検証期間全体での80%カバー率は図のタイトルに出ています（約0.8）。
天気が不安定な日（実測が大きく動く日）は帯が広がり、安定した日は狭くなる傾向があります。

In [ ]:
OUT = ROOT / "outputs"
if not (OUT / "phase5_E1_0.csv").exists():
    raise SystemExit("先に .venv/bin/python src/run_phase5.py を実行してください（約10分）")
e1 = pd.read_csv(OUT / "phase5_E1_0.csv")
e1

**出力の見方**（L=1・4世界の平均）:
- **平均pinball損失**（小さいほど良い）: **GRU 0.295** ＜ MLP 0.300 ＜ HGB（全特徴量）0.307 ≒ HGB（予報・季節だけ）0.310 ＜ 線形＋一定幅 **0.339**。ニューラルネットが最良ですが、GRUとMLPの差（0.005）は標準誤差（0.002）の2倍ほどで、僅差です。**線形＋一定幅より、約1割良い**。
- **80%区間のカバー率**: 線形＋一定幅 0.814、**MLP 0.809、GRU 0.803**（0.8に近く、**正直**）。**HGBは 0.767（全特徴量）・0.781（予報・季節だけ）と低い**（自信過剰）。
- **実測が10%点以下だった割合**（理想は0.10）: MLP 0.088、GRU 0.091、線形 0.090 に対し、**HGB（全特徴量）は 0.121**。**90%点以下だった割合**（理想は0.90）: HGB 0.888、MLP 0.897。HGBは**両側とも、区間が内側に寄りすぎ**ています。
- 区間の幅は、HGBが最も狭い（2.44）が、それは**狭すぎるから**です。**カバー率が足りない狭さは、良さではありません**。MLP・GRUは、線形（2.85）より**約10〜13%狭い**幅で、**同じカバー率**を達成している。これが「情報がある」ということです。

In [ ]:
e2 = pd.read_csv(OUT / "phase5_E2_0.csv")
e2

**出力の見方**（学習期間の最初の8年で学習し、最後の2年で校正。val は未使用）:
- **HGB（全特徴量）**: 校正期間でも、カバー率が **0.750**（目標0.8に届かない）。**学習した期間では当たっていても、新しい期間で外れる**典型的な過学習。CQRで区間を両側に **0.117 kWh** 広げると、val のカバー率は **0.761 → 0.812** に回復します（幅は 2.44 → 2.67）。
- **MLP**: 校正前から 0.812 と正直。広げる量は **0.015** と小さい。校正後 0.819。
- **GRU**: 校正前 0.808、広げる量 0.041、校正後 0.824。
- 校正すると、HGBも **MLP・GRUと同じ程度のカバー率**になるが、幅は HGB 2.67 ＞ MLP 2.62 ＞ GRU 2.57 で、**ニューラルネットのほうが狭い**。
- 示唆: **精度の指標（MAE）だけでなく、区間の正直さ（カバー率）を確認し、外れていれば校正する**。校正用データは、**学習に使っていない新しい期間**から取る。

In [ ]:
e3a, e3b = pd.read_csv(OUT / "phase5_E3_0.csv"), pd.read_csv(OUT / "phase5_E3_1.csv")
display(e3a)
display(e3b)

**出力の見方**:

**リードごと**（上の表）:
- 区間の幅は、**リードが長いほど広がります**: 線形＋一定幅は 2.85（L=1）→ 5.46（L=3）→ 8.59（L=5）、MLPは 2.55 → 4.65 → 8.11。**先の日ほど不確かさが大きい**という、直感どおりの結果です。
- 平均pinball損失もMLPが良い（L=1: 0.300 対 0.339、L=3: 0.556 対 0.672、L=5: 0.955 対 1.062）。差は L=3 で最大です。
- カバー率は L=5 で、線形 0.785、MLP 0.796 と、やや低め。**リードが長いと、区間がわずかに自信過剰**になる傾向があります。

**天気ごと**（下の表・L=1）:
- 線形＋一定幅は、**天気によらず幅が同じ（2.852）**。晴れ0.830・曇り0.812・雨0.798と、カバー率はほぼ0.8の周りです。
- **MLPは、天気（予報の値）によって幅を変えます**: 晴れ **2.19**（一定幅より約23%狭い）、曇り **2.87**（最も広い）、雨 2.67。曇りの日は予報が最も当てにならないためです。
- ただし**曇りのカバー率は 0.776**と、目標より低い。幅を変える分、**条件付きのカバー率**はばらつく。**全体で0.8でも、天気ごとに見ると外れている**ことがあります。

## 試してみよう（演習）
1. **区間の水準を変える**: `LO, HI = 1, 7`（20%点〜80%点＝60%区間）に変えて、カバー率が0.6の周りになるか確かめてください。
2. **校正後の天気別カバー率**: E2 のCQR（区間を一定量広げる）を行ったあと、天気ごとのカバー率を計算してください。分かるのは、一定量の校正では、曇りの外れは直らないことです。
3. **分位点を減らす**: `QS = [0.1, 0.5, 0.9]` にして、pinball損失とカバー率が変わるか見てください。
4. **ノイズを増やす**: `make_series(..., gen_noise=0.08)` の世界で、区間の幅がどう変わるか見てください。

## まとめ
- **pinball loss** で、MLP・GRU・勾配ブースティングに分位点を学習させられる。
- **ニューラルネット（MLP・GRU）の分位点予測**は、カバー率が0.8に近く（0.803〜0.809）、**線形＋一定幅より約1割良い**（pinball 0.295〜0.300 対 0.339）。**HGBは自信過剰**（カバー率 0.767）だが、**CQR** で 0.812 に直せる。
- 区間の幅は、**リードが長いほど広がる**（L=1→5で約3倍）。ニューラルネットは、**天気によって幅を変える**（晴れで狭く、曇りで広い）。
- **カバー率は、全体だけでなく、条件ごと（天気など）にも見る**。

次の Phase6 では、**最終評価**です。設定を開発用の世界で固定し、新しい世界で1回だけ評価し、さらに**実データ**（手元のみ）に適用して、最終レポートをまとめます。